# 02 — Model Training
Fine-tune pretrained ResNet-18 on the face mask dataset.

**Runtime:** Set to GPU (T4) before running — Runtime → Change runtime type → T4 GPU

In [ ]:
import kagglehub
path = kagglehub.dataset_download('vijaykumar1799/face-mask-detection')
print(f'Dataset at: {path}')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

# --- Config ---
DATA_DIR   = path   # adjust if class folders are in a subfolder
IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 15
LR         = 1e-4
VAL_SPLIT  = 0.15
TEST_SPLIT = 0.15
SEED       = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
torch.manual_seed(SEED)
CLASS_NAMES = ['mask_weared_incorrect', 'with_mask', 'without_mask']

## Data Loading & Augmentation
Training set uses augmentation (flips, color jitter, rotation). Val/test use only resize and normalize.

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(DATA_DIR)
n_total = len(full_dataset)
n_test  = int(n_total * TEST_SPLIT)
n_val   = int(n_total * VAL_SPLIT)
n_train = n_total - n_val - n_test

train_set, val_set, test_set = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset.dataset.imgs[self.subset.indices[idx]]
        from PIL import Image
        img = Image.open(img).convert('RGB')
        return self.transform(img), label

train_set.dataset.transform = train_transforms
val_set_wrapped  = TransformSubset(val_set,  eval_transforms)
test_set_wrapped = TransformSubset(test_set, eval_transforms)

train_loader = DataLoader(train_set,         batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set_wrapped,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_set_wrapped,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {n_train} | Val: {n_val} | Test: {n_test}')

## Model Setup
ResNet-18 pretrained on ImageNet. Early layers frozen; last residual block and classifier fine-tuned.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for name, param in model.named_parameters():
    if 'layer4' not in name and 'fc' not in name:
        param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 3)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable:,}')

## Training Loop

In [ ]:
def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0, 0, 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += imgs.size(0)
    return total_loss / total, correct / total

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss,   val_acc   = run_epoch(val_loader,   training=False)
    scheduler.step()
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    print(f'Epoch {epoch:02d}/{EPOCHS} | Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}')

print('Training complete.')

## Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'],   label='Val')
ax2.axhline(0.9, color='r', linestyle='--', alpha=0.5, label='90% target')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## Save Model

In [ ]:
import torch
torch.save(model.state_dict(), 'face_mask_classifier.pth')
print('Saved: face_mask_classifier.pth')

# Save test_loader for use in evaluation notebook
import pickle
with open('test_indices.pkl', 'wb') as f:
    pickle.dump(test_set.indices, f)
print('Saved test indices for evaluation notebook.')